In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
# from tdc.multi_pred import DTI

> could not load openbabel


## 0. Imports

In [3]:
## params
active_c = '#008bfb' # "#3B85C1"
silent_c = '#ff0051'

# data
RAW_PROTEOMICS_PATH   = 'data/MS/20260424_Proteomics_Database_CSV_Export.csv' # df_raw
CLEAN_PROTEOMICS_PATH = 'data/MS/20260429_CDD_MS_SilentActive.csv' # silent vs active
CHEMLIB_PATH          = 'data/chemical_libs/20260430_SERAC_lib.csv' # smiles + compound
OT_ROOT               = 'data/external/opentarget'
PHARMA_PATENT_CSV     = 'data/patent/20260512_pharma_sm.csv' # pharma targets of interest
PX_SCREEN_LIB         = 'data/MS/20260513_CDD_FBXO31_PxScreen_Source.csv'
# output
OT_CACHE              = 'output/MS/opentargets_target_disease.parquet'
GENE_SAR_OUT          = 'output/MS/20260509_geneSAR_R2_full_genome.csv' # R2 per gene
MCS_CSV               = 'output/MS/20260505_target_final_mcs.csv' # MCS enrichment
ML_MODEL_OUTPUT       = 'output/ML/trained_models/20260513' # dump for trained ML models
# dropbox/system
PATENTS_RAW           = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/5_Literature and Patents/0_Companies/Pharma_SmallMolecule_Patents_MASTER.xlsx'
DROPBOX_PROT          = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/12_Proteomics/5_Inventory/CDDVault/Proteomics/GiorgioTamo/'
DROPBOX_ML            = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/'
DOWNLOADS             = '/mnt/c/Users/gtamo/Downloads/'
ENAMINE_20260513      = DROPBOX_ML+'virtual libraries/enum_NAr-pyrimidine_Enamine 4.sdf'

# misc
CM2RM                 = ['SRB-0005653']
FEATURES_TYPE         = 'prevalence' # 'autoresearch' # 


### 20260528 - Prioritizing virtual & library compounds vs PCSK9
We will use the 20260513 trained models to proritize compounds coming from a virtual enumeration and unscreen compounds from our library

In [4]:
## get enamine virtual enum - convert sdf to smiles and extract relevant fields/columns
enum_df = rdkit_tools.get_smiles_df_from_enum(ENAMINE_20260513)
enum_df['compound']  = enum_df['R1_Code'] + '_' + enum_df['R2_Code']
enum_df['filename'] = Path(ENAMINE_20260513).stem
enum_df['smiles'] = enum_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum_df.head(1)

100%|██████████| 9441/9441 [00:03<00:00, 2860.29it/s]


,smiles,R1_Code,R2_Code,compound,filename
0,CCC(CC)(CC)CNc1nccc(-c2ccc3c(c2)[C@@H]2CNC(=O)...,EN300-106990,EN300-53215384,EN300-106990_EN300-53215384,enum_NAr-pyrimidine_Enamine 4


In [5]:
## get unscreened library:
lib = pd.read_csv(PX_SCREEN_LIB).rename(columns={'Molecule Name':'compound','SMILES':'smiles','Lib ID':'lib_id','Px_screened_source':'source'})
lib['smiles'] = lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
print(lib['Px_screened_anywhere'].unique())
lib['Px_sreened'] = 0
lib.loc[lib['Px_screened_anywhere']=='yes','Px_sreened'] = 1
lib = lib[['compound','smiles','lib_id','source','Px_sreened']]

# select uncreened compounds from enamine 4 lib id
# lib = lib[ (lib['lib_id']=='Enamine 4') & (lib['Px_sreened']==0) ]
# print(lib.shape)
# lib.head(1)

# get all the screened compounds:
Px_screened_smiles = list(lib[lib['Px_sreened']==1]['smiles'])
len(Px_screened_smiles)

  0%|          | 0/6444 [00:00<?, ?it/s]

100%|██████████| 6444/6444 [00:01<00:00, 3444.80it/s]

<ArrowStringArray>
[nan, 'yes']
Length: 2, dtype: str


5097

In [6]:
## remove anything that could have been screened from the enumeration:
print(f"before {enum_df.shape}")
enum_df = enum_df[~enum_df['smiles'].isin(Px_screened_smiles)]
print(f"after {enum_df.shape}")

before (9441, 5)
after (9181, 5)


In [ ]:
## prioritize compounds:
model_path = 'output/ML/trained_models/20260513/PCSK9_RF_H236.joblib'

def MLReg_prioritize_compounds(ori_data, model, top=300, features_n=None):
    """
    Predict per-compound labels with a trained joblib RF model, then return
    the top-N most active rows from `ori_data` (most-negative `predicted_label`).

    The prevalence cut (if any) is applied automatically via the saved
    `feature_cols` — we compute the FULL H236 feature universe on `ori_data`
    and let column selection pick exactly the bits the model was trained on.

    :param df ori_data:  DataFrame with at least `compound` and `smiles` columns.
    :param str model:    path to a joblib bundle (dict with 'model' + 'feature_cols')
                         or a bare sklearn estimator.
    :param int top:      how many top compounds to return (sorted by predicted_label asc).
    :param str features_n: name of the featurizer to use; overrides the bundle's
                         saved `features` field. Defaults to None → use the bundle's
                         value, falling back to 'H236' if absent.
    :return df: `ori_data` rows with an added `predicted_label` column, sliced to top-N.
    """
    # Both 'H236' and 'prevalence' use the same featurizer — the prevalence
    # cut lives in the bundle's saved `feature_cols`, not in the featurizer itself.
    features2fn = {
        'H236':       rdkit_tools.compute_H236_features,
    }

    # 1) Load model bundle (dict with model + feature_cols, or a bare estimator)
    bundle = joblib.load(model)
    if isinstance(bundle, dict):
        rf             = bundle['model']
        feature_cols   = bundle['feature_cols']
        gene           = bundle.get('gene', '?')
        R2             = bundle.get('cv_r2')
        mod_features_n = bundle.get('features', 'H236')
    else:
        rf, feature_cols, gene, R2, mod_features_n = bundle, None, '?', None, 'H236'

    # 2) Compute features (full universe; column selection downstream applies
    #    the training-time prevalence cut implicitly).
    feat_fn = features2fn[features_n or mod_features_n]
    feat_df = feat_fn(ori_data)
    print(f"feat_df shape before feature selection: {feat_df.shape}")

    # 3) Predict — preserve the exact column order the model was trained on.
    X = feat_df[feature_cols] if feature_cols is not None else feat_df.drop(columns=['compound'])
    feat_df['predicted_label'] = rf.predict(X)

    # 4) Merge predictions back, sort ascending (most-negative = strongest down-modulator), take top
    out = (ori_data
           .merge(feat_df[['compound', 'predicted_label']], on='compound', how='left')
           .sort_values('predicted_label', ascending=True)
           .head(top)
           .reset_index(drop=True))

    print(f"> {gene}  R²={R2}  features={mod_features_n}  "
          f"({len(feature_cols) if feature_cols is not None else '?'} cols)  "
          f"predicted {feat_df.shape[0]:,}/{len(ori_data):,} compounds; top {top}")
    return out


## get predicted downmodulation
pred_df = MLReg_prioritize_compounds(ori_data=enum_df, model=model_path, top=300, features_n=None)


feat_df shape before feature selection: (9181, 4270)
> PCSK9  R²=0.124  features=H236  (2589 cols)  predicted 9,181/9,181 compounds; top 300


In [35]:
## write sdf in dropbox folder:
if pred_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred_df,
        source_dir=DROPBOX_ML+'virtual libraries', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf',
        pred_col='predicted_label',
)

In [7]:
pred_df.head(3)

NameError: name 'pred_df' is not defined